# 28 — Multi-View, Multi-Frame, and Study-Level Ultrasound Learning

Real ultrasound examinations often contain many frames, views, and cine loops. In this notebook we move from one-image classification to **study-level learning**.

We will study:

- Multiple images per patient
- Multiple views per examination
- Cine-loop and frame sequences
- Image-level vs study-level modeling
- Mean/max probability aggregation
- Majority voting
- Learned feature aggregation
- Attention pooling
- Multi-instance learning
- Patient-balanced sampling
- Variable image counts
- Masked batching
- Leakage prevention across frames/studies
- Study-level evaluation


In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)


# 1. Why One Frame May Not Be Enough

A study can contain several complementary views. A single frame may miss a lesion or anatomical boundary.

Image-level prediction:

$$
x_i \rightarrow p_i
$$

Study-level prediction:

$$
\{x_1,\ldots,x_M\}\rightarrow p_{study}
$$


# 2. Leakage Rule

All related frames from the same patient/study must remain in the same split:

$$
\boxed{Patient/Study\ Grouping}
$$


In [ ]:
def make_frame(label, size=40, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(1,size,size,generator=g)*0.08
    c=size//2
    if label==0:
        x[:,6:-6,c-2:c+2]+=1
    elif label==1:
        x[:,c-2:c+2,6:-6]+=1
    else:
        x[:,c-6:c+6,c-6:c+6]+=1
    return x.clamp(0,1)


In [ ]:
frames=[]
studies=[]
for p in range(60):
    label=p%3
    n_frames=2+(p%5)
    idxs=[]
    for f in range(n_frames):
        idxs.append(len(frames))
        frames.append(make_frame(label,seed=1000+p*10+f))
    studies.append({
        "patient_id":f"P{p:03d}",
        "study_id":f"S{p:03d}",
        "label":label,
        "view":["long","transverse","other"][p%3],
        "frame_indices":idxs,
    })

print("Studies:",len(studies),"Frames:",len(frames))


# 3. Patient-Level Split


In [ ]:
ids=[s["patient_id"] for s in studies]
random.Random(42).shuffle(ids)
train_ids=set(ids[:42]); val_ids=set(ids[42:51]); test_ids=set(ids[51:])
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
print(len(train_ids),len(val_ids),len(test_ids))


# 4. Frame Encoder


In [ ]:
class FrameCNN(nn.Module):
    def __init__(self,feature_dim=48,num_classes=3):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)
        )
        self.proj=nn.Linear(32,feature_dim)
        self.head=nn.Linear(feature_dim,num_classes)

    def encode(self,x):
        x=self.features(x)
        return self.proj(torch.flatten(x,1))

    def forward(self,x):
        return self.head(self.encode(x))


# 5. Probability Aggregation

Mean:

$$
p_{study}=\frac{1}{M}\sum_m p_m
$$

Max:

$$
p_{study,c}=\max_m p_{m,c}
$$


In [ ]:
def mean_probability_aggregation(probabilities):
    return probabilities.mean(dim=0)

def max_probability_aggregation(probabilities):
    x=probabilities.max(dim=0).values
    return x/x.sum()

def majority_vote(predictions,num_classes=3):
    return int(torch.bincount(predictions,minlength=num_classes).argmax())


# 6. Multi-Instance Learning Intuition

A study is a **bag** of images:

$$
B=\{x_1,\dots,x_M\}
$$

The label is attached to the bag. This is the central multi-instance learning (MIL) idea.


In [ ]:
class StudyDataset(Dataset):
    def __init__(self,studies,frames,allowed_ids):
        self.studies=[s for s in studies if s["patient_id"] in allowed_ids]
        self.frames=frames

    def __len__(self):
        return len(self.studies)

    def __getitem__(self,index):
        s=self.studies[index]
        x=torch.stack([self.frames[i] for i in s["frame_indices"]])
        return x,torch.tensor(s["label"]),s["patient_id"]


# 7. Variable-Length Batches

Studies have different numbers of frames. Pad to the maximum frame count and create a mask.

$$
Frames:(N,M_{max},C,H,W)
$$

$$
Mask:(N,M_{max})
$$


In [ ]:
def study_collate_fn(batch):
    xs,ys,ids=zip(*batch)
    m=max(x.shape[0] for x in xs)
    c,h,w=xs[0].shape[1:]
    padded=torch.zeros(len(xs),m,c,h,w)
    mask=torch.zeros(len(xs),m,dtype=torch.bool)
    for i,x in enumerate(xs):
        padded[i,:x.shape[0]]=x
        mask[i,:x.shape[0]]=True
    return padded,mask,torch.stack(ys),list(ids)


In [ ]:
ds=StudyDataset(studies,frames,train_ids)
loader=DataLoader(ds,batch_size=8,shuffle=True,collate_fn=study_collate_fn)
bx,bm,by,bids=next(iter(loader))
print(bx.shape,bm.shape,by.shape)


# 8. Masked Mean Feature Pooling


In [ ]:
def masked_mean(features,mask):
    m=mask.unsqueeze(-1).float()
    return (features*m).sum(dim=1)/m.sum(dim=1).clamp_min(1)


# 9. Mean-Pooling Study Model


In [ ]:
class MeanStudyModel(nn.Module):
    def __init__(self,feature_dim=48,num_classes=3):
        super().__init__()
        self.encoder=FrameCNN(feature_dim,num_classes)
        self.classifier=nn.Linear(feature_dim,num_classes)

    def forward(self,frames,mask):
        n,m,c,h,w=frames.shape
        z=self.encoder.encode(frames.view(n*m,c,h,w)).view(n,m,-1)
        pooled=masked_mean(z,mask)
        return self.classifier(pooled)


In [ ]:
model=MeanStudyModel()
print(model(bx,bm).shape)


# 10. Attention-Based Pooling

Learn frame importance:

$$
z_{study}=\sum_m a_mz_m,\qquad \sum_m a_m=1
$$


In [ ]:
class AttentionPool(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.score=nn.Sequential(nn.Linear(dim,32),nn.Tanh(),nn.Linear(32,1))

    def forward(self,z,mask):
        scores=self.score(z).squeeze(-1)
        scores=scores.masked_fill(~mask,-1e9)
        weights=torch.softmax(scores,dim=1)
        pooled=(z*weights.unsqueeze(-1)).sum(dim=1)
        return pooled,weights


In [ ]:
class AttentionStudyModel(nn.Module):
    def __init__(self,feature_dim=48,num_classes=3):
        super().__init__()
        self.encoder=FrameCNN(feature_dim,num_classes)
        self.pool=AttentionPool(feature_dim)
        self.classifier=nn.Linear(feature_dim,num_classes)

    def forward(self,frames,mask,return_attention=False):
        n,m,c,h,w=frames.shape
        z=self.encoder.encode(frames.view(n*m,c,h,w)).view(n,m,-1)
        pooled,wts=self.pool(z,mask)
        logits=self.classifier(pooled)
        return (logits,wts) if return_attention else logits


In [ ]:
attn_model=AttentionStudyModel()
logits,wts=attn_model(bx,bm,return_attention=True)
print(logits.shape,wts.shape)


# 11. Attention Is Not Causality

Attention weights tell us how the pooling mechanism weighted frames. They do not prove that those frames contain causal evidence.


# 12. Cine Loops

For ordered cine frames, possible models include:

- CNN + mean pooling
- CNN + LSTM/GRU
- 3D CNN
- Temporal transformer

Use temporal order only when it is clinically meaningful.


# 13. Patient-Balanced Sampling

If one patient has 30 frames and another has 3, image-level sampling heavily overweights the first patient.

Study-level sampling naturally gives each study one draw.


# 14. Training Loop


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_epoch(model,loader,optimizer,criterion,device):
    model.train()
    total=0.0;n=0
    for x,mask,y,_ in loader:
        x=x.to(device);mask=mask.to(device);y=y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits=model(x,mask)
        loss=criterion(logits,y)
        loss.backward();optimizer.step()
        total+=loss.item()*y.size(0);n+=y.size(0)
    return total/n


# 15. Study-Level Evaluation


In [ ]:
def study_accuracy(model,loader,device):
    model.eval();correct=0;n=0
    with torch.inference_mode():
        for x,mask,y,_ in loader:
            x=x.to(device);mask=mask.to(device);y=y.to(device)
            pred=model(x,mask).argmax(dim=1)
            correct+=(pred==y).sum().item();n+=y.size(0)
    return correct/n


# 16. Mean vs Max vs Attention

$$
\begin{array}{|c|c|}
\hline
Mean & Equal\ frame\ contribution \\
\hline
Max & One\ strong\ frame\ dominates \\
\hline
Attention & Learned\ frame\ weighting \\
\hline
\end{array}
$$


# 17. Common Mistakes

- Frame leakage across splits
- Cine frames split independently
- Padding without masks
- Reporting image-level metrics for a study-level task
- Treating attention weights as proof
- Ignoring frame-count imbalance


# 18. Research Protocol

Compare under identical patient folds/seeds:

1. Image-level model
2. Mean probability aggregation
3. Max probability aggregation
4. Mean feature pooling
5. Attention MIL

Report patient/study-level metrics.


# 19. Exercises

1. Implement max aggregation.
2. Implement majority voting.
3. Create a variable-length study dataset.
4. Write a padded collate function.
5. Implement masked mean pooling.
6. Implement attention pooling.
7. Compare mean and attention pooling.
8. Visualize attention weights.
9. Explain cine-loop leakage.
10. Design a patient-balanced evaluation.


# 20. Key Takeaways

The study-level transition is:

$$
\boxed{
Frames
\rightarrow
Frame\ Encoder
\rightarrow
Aggregation
\rightarrow
Study\ Prediction
}
$$

The key implementation tools are:

$$
\boxed{
Padding+Masking
}
$$

and the key research rule remains:

$$
\boxed{
Related\ Frames/Studies\ Stay\ Together
}
$$


# Next Notebook

# 29 — Vision Transformers and Hybrid CNN–Transformer Models

In the next notebook, we will study patches, self-attention, ViTs, pretrained transformers, grayscale ultrasound adaptation, and hybrid CNN–Transformer models.
